# JS09 Tugas Lab 2: Pencarian Track Terdekat pada Dataset Micro Spotify

Dataset yang dipakai adalah *Spotify Songs with Attributes and Lyrics* (960K lagu) seperti pada tautan Kaggle di modul. File CSV sudah tersedia di:

```text
JS09/dataset/spotify_songs.csv
```

Isinya 223.823 track dari split *train*, hanya kolom nama track dan sembilan fitur audio numerik yang dipertahankan. Kolom lirik sengaja dibuang karena ukurannya sangat besar dan tidak dipakai untuk perhitungan jarak.

Hanya fitur audio numerik yang dipakai, kemudian distandarkan dengan `StandardScaler` agar `tempo` dan `loudness` tidak mendominasi jarak. Exact NN, ANNOY, FAISS, dan HNSW dibandingkan pada query yang sama.

## 1. Import Library dan Load Dataset

`usecols` dipakai supaya kolom lirik yang besar tidak ikut dimuat ke memori.

In [1]:
import os

import pandas as pd
import numpy as np
import time
import faiss
from annoy import AnnoyIndex
import hnswlib
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

# -------------------------------
# Load dataset
# -------------------------------
path = 'dataset/spotify_songs.csv'  # ganti path sesuai lokasi file

if not os.path.exists(path):
    raise FileNotFoundError(
        f"{path} tidak ditemukan. Simpan CSV dataset Spotify di folder JS09/dataset/."
    )

df = pd.read_csv(path)

features = ['danceability', 'energy', 'loudness', 'speechiness', 
            'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']
X = df[features].values

# Standarisasi fitur
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X).astype('float32')

k = 10  # jumlah nearest neighbors

In [2]:
print('Jumlah track:', len(df))
df[features].describe().T[['mean', 'std', 'min', 'max']]

Jumlah track: 223823


,mean,std,min,max
danceability,0.547725,0.171518,0.0,0.993
energy,0.633793,0.242135,0.0,1.000
loudness,-8.228303,3.920735,-60.0,3.744
speechiness,0.084407,0.097872,0.0,0.966
acousticness,0.293773,0.316566,0.0,0.996
instrumentalness,0.081571,0.212085,0.0,1.000
liveness,0.221902,0.200835,0.0,1.000
valence,0.480593,0.248892,0.0,1.000
tempo,121.696407,29.562630,0.0,246.000


## 2. Query dan Exact NN

Query diambil dari track pertama. Exact NN memakai `NearestNeighbors` dengan algoritma brute force sebagai acuan kebenaran.

In [3]:
REPEATS = 20  # satu query terlalu cepat untuk diukur sekali

query_idx = 0
query = X_scaled[query_idx]

nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean')
nn.fit(X_scaled)

start = time.time()
dist_exact, idx_exact = nn.kneighbors(query.reshape(1, -1))
time_exact = time.time() - start

print('Query:', df['track_name'].iloc[query_idx])
print('Exact NN:', idx_exact[0])

Query: When I Survey - The Wondrous Cross
Exact NN: [     0 134781 113026 161616  13155 111227  50573 218822 185061 223508]


## 3. ANNOY

In [4]:
ann = AnnoyIndex(X_scaled.shape[1], 'angular')
for i, v in enumerate(X_scaled):
    ann.add_item(i, v)

start = time.time()
ann.build(10)
time_ann_build = time.time() - start

start = time.time()
for _ in range(REPEATS):
    idx_ann = ann.get_nns_by_vector(query, k)
time_ann = (time.time() - start) / REPEATS

print('ANNOY NN:', idx_ann)

ANNOY NN: [0, 199345, 27420, 161616, 124377, 113026, 185061, 134781, 111227, 218822]


## 4. FAISS

In [5]:
faiss_index = faiss.IndexFlatL2(X_scaled.shape[1])

start = time.time()
faiss_index.add(X_scaled)
time_faiss_build = time.time() - start

start = time.time()
for _ in range(REPEATS):
    _, idx_faiss = faiss_index.search(query.reshape(1, -1), k)
time_faiss = (time.time() - start) / REPEATS

print('FAISS NN:', idx_faiss[0])

FAISS NN: [     0 134781 113026 161616  13155 111227  50573 218822 185061 223508]


## 5. HNSW

In [6]:
hnsw = hnswlib.Index(space='l2', dim=X_scaled.shape[1])
hnsw.init_index(max_elements=len(X_scaled), ef_construction=200, M=16)

start = time.time()
hnsw.add_items(X_scaled)
time_hnsw_build = time.time() - start

hnsw.set_ef(100)

start = time.time()
for _ in range(REPEATS):
    idx_hnsw, _ = hnsw.knn_query(query, k=k)
time_hnsw = (time.time() - start) / REPEATS

print('HNSW NN:', idx_hnsw[0])

HNSW NN: [     0 134781 113026 161616  13155 111227  50573 218822 185061 223508]


## 6. Perbandingan Hasil

Recall dihitung terhadap hasil exact NN. Karena data Spotify punya banyak track dengan karakter audio mirip, tetangga yang berbeda belum tentu salah secara musikal, tetapi tetap dihitung sebagai tidak cocok.

In [7]:
def recall(pred, gt, k):
    return len(set(list(pred)[:k]) & set(list(gt)[:k])) / k

summary = pd.DataFrame([
    {'method': 'Exact NN', 'build_s': 0.0, 'query_ms': time_exact * 1000,
     'recall@10': 1.0},
    {'method': 'ANNOY', 'build_s': time_ann_build, 'query_ms': time_ann * 1000,
     'recall@10': recall(idx_ann, idx_exact[0], k)},
    {'method': 'FAISS', 'build_s': time_faiss_build, 'query_ms': time_faiss * 1000,
     'recall@10': recall(idx_faiss[0], idx_exact[0], k)},
    {'method': 'HNSW', 'build_s': time_hnsw_build, 'query_ms': time_hnsw * 1000,
     'recall@10': recall(idx_hnsw[0], idx_exact[0], k)},
]).round(4)

summary

,method,build_s,query_ms,recall@10
0,Exact NN,0.0000,1720.2291,1.0
1,ANNOY,2.8708,0.0491,0.7
2,FAISS,0.0085,2.3354,1.0
3,HNSW,15.3355,0.1267,1.0


In [8]:
comparison = pd.DataFrame({
    'exact': [df['track_name'].iloc[i] for i in idx_exact[0]],
    'annoy': [df['track_name'].iloc[i] for i in idx_ann],
    'faiss': [df['track_name'].iloc[i] for i in idx_faiss[0]],
    'hnsw': [df['track_name'].iloc[i] for i in idx_hnsw[0]],
})

comparison

,exact,annoy,faiss,hnsw
0,When I Survey - The Wondrous Cross,When I Survey - The Wondrous Cross,When I Survey - The Wondrous Cross,When I Survey - The Wondrous Cross
1,Is There Something in the Movies?,數不盡的星空,Is There Something in the Movies?,Is There Something in the Movies?
2,Apples Dipped in Candy,Tea and Thorazine,Apples Dipped in Candy,Apples Dipped in Candy
3,A Way to You Again,A Way to You Again,A Way to You Again,A Way to You Again
4,Come Away With Me,Sunburn - Deluxe Edition,Come Away With Me,Come Away With Me
5,My Way,Apples Dipped in Candy,My Way,My Way
6,Away in a Manger - 2004 Remaster,Khaki & Corduroy,Away in a Manger - 2004 Remaster,Away in a Manger - 2004 Remaster
7,Red-Winged Blackbird,Is There Something in the Movies?,Red-Winged Blackbird,Red-Winged Blackbird
8,Khaki & Corduroy,My Way,Khaki & Corduroy,Khaki & Corduroy
9,Body And Soul,Red-Winged Blackbird,Body And Soul,Body And Soul


Pada dataset nyata, FAISS dan HNSW mengembalikan tetangga yang sama dengan exact NN, sedangkan ANNOY memakai metric `angular` dengan `search_k` bawaan sehingga recall-nya di bawah 1. Selisih itu wajar: ketiga pustaka ANN hanya menelusuri sebagian ruang vektor, dan itu adalah kompromi yang ditukar dengan waktu query yang jauh lebih cepat.